#Simulation

In [9]:
# ==========================================
# 1. Imports
# ==========================================
from simulator import Config, Simulator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvgis

plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

In [ ]:
# ==========================================
# FACTIBILIDAD + PARETO POR CIUDAD Y CONSUMO
# ==========================================

# Ciudades con TMY obtenido automáticamente desde PVGIS
cities = {
    "Ushuaia":      pvgis.get_tmy_csv("Ushuaia", 2005, 2023),
    "MdP":          pvgis.get_tmy_csv("Mar del Plata", 2005, 2023),
    "Johannesburg": pvgis.get_tmy_csv("Johannesburg", 2005, 2023),
    "Lima":         pvgis.get_tmy_csv("Lima", 2005, 2023),
    "Quito":        pvgis.get_tmy_csv("Quito", 2005, 2023),
    "Bangkok":      pvgis.get_tmy_csv("Bangkok", 2005, 2023),
    "Delhi":        pvgis.get_tmy_csv("New Delhi", 2005, 2023),
    "Cairo":        pvgis.get_tmy_csv("Cairo", 2005, 2023),
    "CR":           pvgis.get_tmy_csv("Almagro", 2005, 2023),
    "Oslo":         pvgis.get_tmy_csv("Oslo", 2005, 2023),
    "Anchorage":    pvgis.get_tmy_csv("Anchorage", 2005, 2023),
}
latitudes = {
    "Ushuaia": -54.8,
    "MdP": -38.0,
    "Johannesburg": -26.2,
    "Lima": -12.0,
    "Quito": -0.2,
    "Bangkok": 13.7,
    "Delhi": 28.6,
    "Cairo": 30.0,
    "CR": 38.99,
    "Oslo": 59.9,
    "Anchorage": 61.2,
}

power_profiles = [0.5, 0.05, 0.005]

pareto_objectives = [
    ("C_batt_Ah", -1),
    ("panel_area_m2", -1),
    ("soc_full_fraction", -1),
]

rows = []

for city, production_data in cities.items():
    for power in power_profiles:
        print(f"Running {city} | {power} W")

        config = Config()
        config.NODE_POWER_W = power
        simulator = Simulator(config)

        results = simulator.run_full_simulation(production_data)
        summary = results["summary"].copy()

        feasible = summary[
            (summary["failure_hours"] == 0) &
            (summary["I_req_max_A"] <= summary["I_batt_max_A"])
        ].copy()

        front = simulator.pareto_front(feasible, pareto_objectives)

        rows.append({
            "City": city,
            "Latitude": latitudes[city],
            "Power_W": power,
            "FeasibleConfigs": len(feasible),
            "ParetoConfigs": len(front),
        })

df_fact = pd.DataFrame(rows).sort_values(["Latitude", "Power_W"])
df_fact

INFO:pvgis:CSV already exists, skipping download: raw-data/Almagro_2005_2023.csv
INFO:pvgis:Coordinates for Mar del Plata: lat=-37.9976168, lon=-57.5482079
INFO:pvgis:{
  "location": {
    "latitude": -37.9976168,
    "longitude": -57.5482079,
    "elevation": 13.0,
    "irradiance_time_offset": 0.5
  },
  "meteo_data": {
    "radiation_db": "PVGIS-ERA5",
    "meteo_db": "ERA5",
    "year_min": 2005,
    "year_max": 2023,
    "use_horizon": true,
    "horizon_db": "DEM-calculated"
  }
}
INFO:pvgis:CSV already exists, skipping download: raw-data/Quito_2005_2023.csv


Running CR | 0.5 W
Loading irradiance data...
Computing PV power...
Computing hourly balance...
Simulating battery SoC...
Evaluating viability...
Computing optimal scores...
Done!
Running CR | 0.05 W
Loading irradiance data...
Computing PV power...
Computing hourly balance...
Simulating battery SoC...
Evaluating viability...
Computing optimal scores...
Done!
Running CR | 0.005 W
Loading irradiance data...
Computing PV power...
Computing hourly balance...
Simulating battery SoC...
Evaluating viability...
Computing optimal scores...
Done!
Running MdP | 0.5 W
Loading irradiance data...
Computing PV power...
Computing hourly balance...
Simulating battery SoC...
Evaluating viability...
Computing optimal scores...
Done!
Running MdP | 0.05 W
Loading irradiance data...
Computing PV power...
Computing hourly balance...
Simulating battery SoC...
Evaluating viability...
Computing optimal scores...
Done!
Running MdP | 0.005 W
Loading irradiance data...
Computing PV power...
Computing hourly balanc

,City,Latitude,Power_W,FeasibleConfigs,ParetoConfigs
5,MdP,-38.00,0.005,2005,13
4,MdP,-38.00,0.050,910,13
3,MdP,-38.00,0.500,0,0
8,Quito,-0.20,0.005,2226,3
7,Quito,-0.20,0.050,1419,8
6,Quito,-0.20,0.500,18,1
2,CR,38.99,0.005,2120,12
1,CR,38.99,0.050,1044,12
0,CR,38.99,0.500,0,0


In [11]:
df_table = df_fact.pivot(
    index=["City", "Latitude"],
    columns="Power_W",
    values=["FeasibleConfigs", "ParetoConfigs"]
)

df_table

FeasibleConfigs             ParetoConfigs            
Power_W                  0.005 0.050 0.500         0.005 0.050 0.500
City  Latitude                                                      
CR     38.99              2120  1044     0            12    12     0
MdP   -38.00              2005   910     0            13    13     0
Quito -0.20               2226  1419    18             3     8     1